# NCTBench-Pilot — Step 1: PDF Audit
Discovers all NCTB textbook PDFs and classifies each as
text-native (PyMuPDF) or image-based (EasyOCR).
Corresponds to Section 3.1 Corpus Construction of the paper.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path(r"E:\CSAA Project\pipeline")))
from config import NCTB_ROOT, MANIFEST_PATH
from utils import get_all_pdfs
print("Config loaded")
print(f"NCTB root : {NCTB_ROOT}")
print(f"Manifest  : {MANIFEST_PATH}")

## Discover PDFs
Filename pattern: {Subject}_{lang}_class{N}.pdf

In [ ]:
pdfs = get_all_pdfs(NCTB_ROOT)
print(f"Total PDFs: {len(pdfs)}\n")
print(f"{'Filename':42s} {'Class':6s} {'Lang':5s} {'Subject':10s}")
print("-" * 67)
for p in pdfs:
    print(f"{p['filename']:42s} "
          f"{p['class_num']:6d} "
          f"{p['lang_code']:5s} "
          f"{p['subject']:10s}")

## Audit Results
Bengali PDFs always use EasyOCR.
English PDFs with sparse embedded text also use EasyOCR.

In [ ]:
from step1_audit import run_audit
run_audit()

## Manifest Inspection

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print(f"Total entries: {len(manifest)}\n")
print(f"{'Filename':42s} {'Status':12s} {'Pages':6s} {'Method':10s} {'MB':6s}")
print("-" * 80)
for fname, e in sorted(manifest.items()):
    print(f"{fname:42s} "
          f"{e['status']:12s} "
          f"{e.get('page_count',0):6d} "
          f"{e.get('extraction_method','?'):10s} "
          f"{e.get('file_size_mb',0):6.1f}")

## Summary Statistics

In [ ]:
from collections import Counter
methods = Counter(
    e.get("extraction_method","?") for e in manifest.values()
)
total_pages = sum(e.get("page_count",0) for e in manifest.values())
total_mb = sum(e.get("file_size_mb",0) for e in manifest.values())
print("Extraction method breakdown:")
for m, c in methods.items():
    print(f"  {m}: {c} PDFs")
print(f"\nTotal pages : {total_pages}")
print(f"Total size  : {total_mb:.1f} MB")